# LSGNN — model training (notebook smoke test)

Use this notebook to:

1. **Environment**: Verify PyTorch, CUDA, and PyG in the current kernel (GPU or CPU).
2. **Code path**: Run **preprocess (if needed) → `train_single_run`**, same logic as `python -m src.models.train`.

**Tip**: Open this file from `notebooks/` inside the repo, or run the project-root cell first so `cwd` is the directory that contains `src/`.

**Quick run**: Set `SMOKE_EPOCHS` small (e.g. `3`) to sanity-check; for full runs use `200` or the value from `src/configs/<dataset>.json`.

In [1]:
import os
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "src").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "src").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(
        f"Project root not found (expected a directory containing src/). cwd={_cwd}. "
        "Open the notebook from the repo root or from notebooks/, then re-run this cell."
    )

os.chdir(REPO_ROOT)
_root_s = str(REPO_ROOT)
if _root_s not in sys.path:
    sys.path.insert(0, _root_s)

print("REPO_ROOT =", REPO_ROOT)
print("cwd       =", Path.cwd())

REPO_ROOT = /home/zhaog30/cas_747/Guanhua_Zhao
cwd       = /home/zhaog30/cas_747/Guanhua_Zhao


## Environment check

If `torch.cuda.is_available()` is `False`, training still runs on CPU but will be slower. For GPU, select a kernel where CUDA-enabled PyTorch is installed.

In [2]:
import importlib

import torch

print("python   ", sys.version.split()[0])
print("torch    ", torch.__version__, "| cuda", torch.version.cuda)
print("cuda ok? ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device   ", torch.cuda.get_device_name(0))

for mod in ("torch_geometric", "torch_sparse", "torch_scatter"):
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print(f"{mod:16s}", ver)
    except Exception as e:
        print(f"{mod:16s}", "IMPORT FAILED:", e)

python    3.11.15
torch     2.11.0+cu128 | cuda 12.8
cuda ok?  True
device    NVIDIA GeForce RTX 5070


/home/zhaog30/miniconda3/envs/lsgnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch_geometric  2.7.0
torch_sparse     0.6.18+pt211cu128
torch_scatter    2.1.2+pt211cu128


## Data: preprocess if `data/processed/<dataset>.pt` is missing

Supported dataset slugs (same set as `CONFIG_DATASETS` in `src/models/train.py`). In the next cell, replace `DATASET` with one of these; use the **same** string for `PROCESSED_NAME` in the training cell unless you use a custom processed filename.

`actor`, `arxiv-year`, `chameleon`, `citeseer`, `cornell`, `cora`, `ogbn-arxiv`, `pubmed`, `squirrel`, `texas`, `wisconsin`

This cell runs `python -m src.data_processing.preprocess` in a subprocess (same as the CLI). If raw data is missing, PyG may download it depending on network and `LSGNN_ALLOW_DOWNLOAD`.

In [5]:
import subprocess

DATASET = "cora"
processed_pt = REPO_ROOT / "data" / "processed" / f"{DATASET}.pt"

if processed_pt.is_file():
    print("found:", processed_pt)
else:
    print("running preprocess...")
    subprocess.run(
        [sys.executable, "-m", "src.data_processing.preprocess", "--dataset", DATASET],
        cwd=REPO_ROOT,
        check=True,
    )
    print("done:", processed_pt)

found: /home/zhaog30/cas_747/Guanhua_Zhao/data/processed/cora.pt


## Training: `train_single_run` (same as `train.py`)

- `DATASET` / `PROCESSED_NAME`: use the same slug as in the preprocess cell (supported names are listed in **Data** above).
- `SMOKE_EPOCHS`: keep small for a smoke test; raise to match `src/configs/<dataset>.json` or set `cfg["num_epochs"]` here.
- `SPLIT_IDX`: same meaning as CLI `--split-idx` when multiple splits exist.
- Checkpoints (`.ckpt`) go under `MODEL_OUTPUT_DIR` (default below), not `results/model_outputs`, so notebook runs do not overwrite CLI checkpoints. Training curve PNGs always go to `results/plots/<dataset>/` (same filename pattern as before: `lsgnn_*_curves.png`). Logs still go to `results/logs/<dataset>/train.log` like the CLI.

In [6]:
from src.data_processing.load_data import sync_ds_old_env_with_config
from src.models.train import (
    CONFIG_DATASETS,
    _normalize_dataset_name,
    default_lsgnn_config,
    load_lsgnn_config_json,
    train_single_run,
)
from src.utils.log_setup import get_logger, setup_logging

DATASET = "cora"
PROCESSED_NAME = "cora"
SPLIT_IDX = 0
SMOKE_EPOCHS = 3

ds = _normalize_dataset_name(DATASET)
cfg = default_lsgnn_config(ds)
if ds in CONFIG_DATASETS:
    cfg.update(load_lsgnn_config_json(ds))
cfg["split_idx"] = SPLIT_IDX
cfg["num_epochs"] = SMOKE_EPOCHS

sync_ds_old_env_with_config(ds)

setup_logging(file_name="train.log", subdir=ds)
log = get_logger("train")

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_OUTPUT_DIR = str(REPO_ROOT / "results" / "notebook_model_outputs")
PLOTS_DIR = str(REPO_ROOT / "results" / "plots")
print(f"train {ds} | epochs={cfg['num_epochs']} | split_idx={SPLIT_IDX} | device={dev}")
print(f"model_dir={MODEL_OUTPUT_DIR}")
print(f"plots_dir={PLOTS_DIR}")

metrics = train_single_run(
    ds, cfg, PROCESSED_NAME, log, model_dir=MODEL_OUTPUT_DIR, plots_dir=PLOTS_DIR
)
metrics

train cora | epochs=3 | split_idx=0 | device=cuda
model_dir=/home/zhaog30/cas_747/Guanhua_Zhao/results/notebook_model_outputs


/home/zhaog30/cas_747/Guanhua_Zhao/src/utils/helpers.py:55: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  return torch.sparse.FloatTensor(indices, values, shape)
train cora_split0:   0%|                        | 0/3 [00:00<?, ?it/s]

2026-04-06 01:39:49 INFO [lsgnn.train] start training cora with config: {'num_epochs': 3, 'train_val_test': [0.48, 0.32, 0.2], 'undirected': True, 'transposed': False, 'hidden_channels': 16, 'lr': 0.01, 'wd': 0.0, 'dropout': 0.9, 'adamw': False, 'K': 5, 'beta': 1.0, 'gamma': 0.7, 'method': 'norm2', 'num_reduce_layers': 1, 'use_A_embedding': False, 'out_norm': True, 'out_mlp': False, 'use_irdc': True, 'split_idx': 0, 'ds_old': False, 'use_tqdm': True, 'save_by_epoch': False}
2026-04-06 01:39:53 INFO [lsgnn.train] epoch 0 train_loss=2.0334 val_loss=1.7072 val_acc=0.4360 test_acc=0.4525


train cora_split0:  33%|█████▎          | 1/3 [00:03<00:07,  3.84s/it]

2026-04-06 01:39:53 INFO [lsgnn.train] epoch 1 train_loss=1.7451 val_loss=1.5971 val_acc=0.4660 test_acc=0.5289
2026-04-06 01:39:53 INFO [lsgnn.train] epoch 2 train_loss=1.6102 val_loss=1.4864 val_acc=0.5479 test_acc=0.6145


train cora_split0: 100%|████████████████| 3/3 [00:03<00:00,  1.29s/it]


2026-04-06 01:39:53 INFO [lsgnn.train] training curves saved to /home/zhaog30/cas_747/Guanhua_Zhao/results/notebook_model_outputs/cora/lsgnn_cora_split0_curves.png
2026-04-06 01:39:53 INFO [lsgnn.train] best checkpoints: loss=/home/zhaog30/cas_747/Guanhua_Zhao/results/notebook_model_outputs/cora/lsgnn_cora_split0_best_loss.ckpt acc=/home/zhaog30/cas_747/Guanhua_Zhao/results/notebook_model_outputs/cora/lsgnn_cora_split0_best_acc.ckpt
2026-04-06 01:39:53 INFO [lsgnn.train] dataset=cora_split0 precompute_wall_time_sec=5.445 training_cost_time_sec=3.844


{'val_acc': 0.5478662252426147,
 'test_acc': 0.6145251393318176,
 'val_acc_at_best_val_loss': 0.5478662252426147,
 'test_acc_at_best_val_loss': 0.6145251393318176,
 'ckpt_path': '/home/zhaog30/cas_747/Guanhua_Zhao/results/notebook_model_outputs/cora/lsgnn_cora_split0_best_loss.ckpt',
 'ckpt_best_loss_path': '/home/zhaog30/cas_747/Guanhua_Zhao/results/notebook_model_outputs/cora/lsgnn_cora_split0_best_loss.ckpt',
 'ckpt_best_acc_path': '/home/zhaog30/cas_747/Guanhua_Zhao/results/notebook_model_outputs/cora/lsgnn_cora_split0_best_acc.ckpt',
 'curves_path': '/home/zhaog30/cas_747/Guanhua_Zhao/results/notebook_model_outputs/cora/lsgnn_cora_split0_curves.png',
 'precompute_wall_time_sec': 5.444978952407837,
 'training_cost_time_sec': 3.8437936305999756}